# Canadian Financial and Monetary Policy Quantification

This notebook shows the raw data, formulas, and step-by-step calculations used to quantify losses to Canadians from Bank of Canada policies, Big Five bank profits, inflation, and the mortgage renewal shock.

**Principle:** Every claim is backed by a source, a formula, and a stated assumption.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

# Set display options
pd.set_option('display.float_format', lambda x: '%.2f' % x)

## 1. Bank of Canada Net Income / Losses

**Source:** Bank of Canada Annual Reports 2019–2025

**Formula:** Net income (loss) = total income − total expenses

In [2]:
boc_results = pd.DataFrame({
    'Year': [2019, 2020, 2021, 2022, 2023, 2024, 2025],
    'Net_Income_Millions': [1302, 1964, 2401, -1111, -5652, -3079, -82],
    'Source': [
        'Bank of Canada Annual Report 2019 ($1.3B net income)',
        'Bank of Canada Annual Report 2020 ($1.96B net income)',
        'Bank of Canada Annual Report 2021 ($2.40B net income)',
        'Bank of Canada Annual Report 2022 ($1.11B net loss)',
        'Bank of Canada Annual Report 2023 ($5.65B net loss)',
        'Bank of Canada Annual Report 2024 ($3.08B net loss)',
        'Bank of Canada Annual Report 2025 ($0.08B net loss)'
    ]
})

boc_results['Cumulative_Net_Income_Millions'] = boc_results['Net_Income_Millions'].cumsum()

print(boc_results)
print(f"\nCumulative BoC net income 2019-2025: ${boc_results['Net_Income_Millions'].sum():,.0f} million = ${boc_results['Net_Income_Millions'].sum()/1000:.2f} billion")
print(f"Cumulative BoC losses 2022-2025: ${boc_results[boc_results['Year'] >= 2022]['Net_Income_Millions'].sum():,.0f} million = ${boc_results[boc_results['Year'] >= 2022]['Net_Income_Millions'].sum()/1000:.2f} billion")

   Year  Net_Income_Millions  \
0  2019                 1302   
1  2020                 1964   
2  2021                 2401   
3  2022                -1111   
4  2023                -5652   
5  2024                -3079   
6  2025                  -82   

                                              Source  \
0  Bank of Canada Annual Report 2019 ($1.3B net i...   
1  Bank of Canada Annual Report 2020 ($1.96B net ...   
2  Bank of Canada Annual Report 2021 ($2.40B net ...   
3  Bank of Canada Annual Report 2022 ($1.11B net ...   
4  Bank of Canada Annual Report 2023 ($5.65B net ...   
5  Bank of Canada Annual Report 2024 ($3.08B net ...   
6  Bank of Canada Annual Report 2025 ($0.08B net ...   

   Cumulative_Net_Income_Millions  
0                            1302  
1                            3266  
2                            5667  
3                            4556  
4                           -1096  
5                           -4175  
6                           -4257  

Cumul

## 2. Big Five Bank Profits

**Source:** Bank annual reports (fiscal years ending October 31)

**Formula:** Total profit = sum of individual bank net income (reported)

In [3]:
bank_profits = pd.DataFrame({
    'Bank': ['RBC', 'TD', 'BMO', 'Scotiabank', 'CIBC'],
    'FY2018': [12.431, 11.334, 5.453, 8.724, 5.284],
    'FY2019': [12.871, 11.686, 5.758, 8.798, 5.121],
    'FY2020': [11.437, 11.895, 5.097, 6.853, 3.792],
    'FY2021': [16.100, 14.298, 7.754, 9.955, 6.446],
    'FY2022': [15.807, 17.429, 13.537, 10.174, 6.243],
    'FY2023': [14.866, 10.782, 4.437, 7.530, 5.000],
    'FY2024': [16.240, 8.842, 7.327, 7.892, 7.154],
})

print(bank_profits)

for year in ['FY2018', 'FY2019', 'FY2020', 'FY2021', 'FY2022', 'FY2023', 'FY2024']:
    total = bank_profits[year].sum()
    print(f"Big Five total profit {year}: ${total:.2f} billion")

print(f"\nTotal Big Five profit 2018-2024: ${bank_profits[['FY2018','FY2019','FY2020','FY2021','FY2022','FY2023','FY2024']].sum().sum():.2f} billion")

         Bank  FY2018  FY2019  FY2020  FY2021  FY2022  FY2023  FY2024
0         RBC   12.43   12.87   11.44   16.10   15.81   14.87   16.24
1          TD   11.33   11.69   11.89   14.30   17.43   10.78    8.84
2         BMO    5.45    5.76    5.10    7.75   13.54    4.44    7.33
3  Scotiabank    8.72    8.80    6.85    9.96   10.17    7.53    7.89
4        CIBC    5.28    5.12    3.79    6.45    6.24    5.00    7.15
Big Five total profit FY2018: $43.23 billion
Big Five total profit FY2019: $44.23 billion
Big Five total profit FY2020: $39.07 billion
Big Five total profit FY2021: $54.55 billion
Big Five total profit FY2022: $63.19 billion
Big Five total profit FY2023: $42.62 billion
Big Five total profit FY2024: $47.45 billion

Total Big Five profit 2018-2024: $334.35 billion


## 3. Big Five Bank Net Interest Income (Interest Earnings)

**Source:** Bank annual reports (Consolidated Statements of Income / MD&A)

**Formula:** Net interest income = interest income from loans, securities and deposits minus interest expense on deposits and borrowings

**Note:** This is the *interest* the banks earned, before provisions for credit losses and non-interest expenses. It is not the same as net income.

In [4]:
bank_nii = pd.DataFrame({
    'Bank': ['RBC', 'TD', 'BMO', 'Scotiabank', 'CIBC'],
    'FY2018': [11.916, 22.239, 11.438, 16.191, 10.065],
    'FY2019': [12.664, 23.821, 12.888, 17.177, 10.551],
    'FY2020': [14.048, 24.497, 13.971, 17.320, 11.044],
    'FY2021': [16.038, 24.131, 14.310, 16.961, 11.459],
    'FY2022': [16.827, 27.353, 15.885, 18.115, 12.641],
    'FY2023': [16.230, 29.944, 18.681, 18.287, 12.825],
    'FY2024': [13.509, 30.472, 19.468, 19.252, 13.695],
})

print(bank_nii)

for year in ['FY2018', 'FY2019', 'FY2020', 'FY2021', 'FY2022', 'FY2023', 'FY2024']:
    total = bank_nii[year].sum()
    print(f"Big Five total net interest income {year}: ${total:.2f} billion")

print(f"\nTotal Big Five net interest income 2018-2024: ${bank_nii[['FY2018','FY2019','FY2020','FY2021','FY2022','FY2023','FY2024']].sum().sum():.2f} billion")

         Bank  FY2018  FY2019  FY2020  FY2021  FY2022  FY2023  FY2024
0         RBC   11.92   12.66   14.05   16.04   16.83   16.23   13.51
1          TD   22.24   23.82   24.50   24.13   27.35   29.94   30.47
2         BMO   11.44   12.89   13.97   14.31   15.88   18.68   19.47
3  Scotiabank   16.19   17.18   17.32   16.96   18.11   18.29   19.25
4        CIBC   10.06   10.55   11.04   11.46   12.64   12.82   13.70
Big Five total net interest income FY2018: $71.85 billion
Big Five total net interest income FY2019: $77.10 billion
Big Five total net interest income FY2020: $80.88 billion
Big Five total net interest income FY2021: $82.90 billion
Big Five total net interest income FY2022: $90.82 billion
Big Five total net interest income FY2023: $95.97 billion
Big Five total net interest income FY2024: $96.40 billion

Total Big Five net interest income 2018-2024: $595.91 billion


## 4. Inflation Decomposition

**Source:** Statistics Canada CPI tables

**Formula:** Component contribution = CPI weight × annual price change

In [5]:
cpi_2022 = pd.DataFrame({
    'Component': ['Shelter', 'Food', 'Transportation', 'Energy', 'Household Operations', 
                  'Health/Personal Care', 'Clothing/Footwear', 'Recreation/Education', 'Alcohol/Tobacco/Cannabis'],
    'Weight': [29.80, 15.94, 16.91, 7.41, 14.50, 4.62, 4.31, 9.29, 4.63],
    'Annual_Change_Pct': [6.9, 8.9, 10.6, 22.5, 4.6, 6.1, 1.8, 3.4, 4.8]
})

cpi_2022['Contribution_pp'] = (cpi_2022['Weight'] / 100) * cpi_2022['Annual_Change_Pct']

print(cpi_2022)
print(f"\nSum of contributions: {cpi_2022['Contribution_pp'].sum():.2f} percentage points")
print(f"Reported 2022 headline CPI: 6.8%")

                  Component  Weight  Annual_Change_Pct  Contribution_pp
0                   Shelter   29.80               6.90             2.06
1                      Food   15.94               8.90             1.42
2            Transportation   16.91              10.60             1.79
3                    Energy    7.41              22.50             1.67
4      Household Operations   14.50               4.60             0.67
5      Health/Personal Care    4.62               6.10             0.28
6         Clothing/Footwear    4.31               1.80             0.08
7      Recreation/Education    9.29               3.40             0.32
8  Alcohol/Tobacco/Cannabis    4.63               4.80             0.22

Sum of contributions: 8.50 percentage points
Reported 2022 headline CPI: 6.8%


## 5. Demand-Side Contribution Estimate

**Baseline:** 2% target inflation

**Assumption:** We test three scenarios for how much of excess inflation (above 2%) came from demand-side stimulus versus supply-side shocks.

In [6]:
inflation = pd.DataFrame({
    'Year': [2019, 2020, 2021, 2022, 2023, 2024, 2025],
    'Actual_CPI': [1.9, 0.7, 3.4, 6.8, 3.9, 2.4, 2.1],
    'Baseline_CPI': [2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0]
})

inflation['Excess_CPI'] = inflation['Actual_CPI'] - inflation['Baseline_CPI']

for share in [0.25, 0.50, 0.75]:
    inflation[f'Demand_Share_{int(share*100)}'] = inflation['Excess_CPI'] * share

print(inflation)

print("\nInterpretation (2022 example):")
print("Excess inflation was 4.8 percentage points above the 2% target.")
print("If demand-side stimulus caused 25% of that excess, it contributed 1.2 pp.")
print("If demand-side stimulus caused 50% of that excess, it contributed 2.4 pp.")
print("If demand-side stimulus caused 75% of that excess, it contributed 3.6 pp.")

   Year  Actual_CPI  Baseline_CPI  Excess_CPI  Demand_Share_25  \
0  2019        1.90          2.00       -0.10            -0.03   
1  2020        0.70          2.00       -1.30            -0.33   
2  2021        3.40          2.00        1.40             0.35   
3  2022        6.80          2.00        4.80             1.20   
4  2023        3.90          2.00        1.90             0.47   
5  2024        2.40          2.00        0.40             0.10   
6  2025        2.10          2.00        0.10             0.03   

   Demand_Share_50  Demand_Share_75  
0            -0.05            -0.08  
1            -0.65            -0.98  
2             0.70             1.05  
3             2.40             3.60  
4             0.95             1.42  
5             0.20             0.30  
6             0.05             0.08  

Interpretation (2022 example):
Excess inflation was 4.8 percentage points above the 2% target.
If demand-side stimulus caused 25% of that excess, it contributed 1.2 p

## 6. Mortgage Renewal Shock

**Source:** Bank of Canada Staff Analytical Note 2025-21

**Assumptions:**
- ~60% of outstanding mortgages renew in 2025–2026
- Average payment increase: 10% for 2025 renewals, 6% for 2026 renewals (BoC baseline)
- Average outstanding mortgage balance: ~$350,000
- Earlier years (2019–2024) had minimal or no payment shock because rates were near pandemic lows

In [7]:
# BoC note: 60% of outstanding mortgages renew in 2025-2026; ~60% of those see increases
# Roughly 30% of all mortgages face higher payments in each of 2025 and 2026

total_mortgages = 6_000_000
avg_balance = 350_000
avg_rate_before = 0.025
avg_rate_after_2025 = 0.035
avg_rate_after_2026 = 0.031

# Monthly payment on $350k at 25-year amortization
def monthly_payment(principal, annual_rate, years=25):
    r = annual_rate / 12
    n = years * 12
    return principal * (r * (1 + r)**n) / ((1 + r)**n - 1)

mortgage_shock = pd.DataFrame({
    'Year': [2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026],
    'Share_of_Mortgages_Facing_Increase': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.30, 0.30],
    'Avg_Rate_Before': [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025, 0.025],
    'Avg_Rate_After': [0.025, 0.025, 0.025, 0.025, 0.025, 0.025, avg_rate_after_2025, avg_rate_after_2026]
})

mortgage_shock['Monthly_Payment_Before'] = mortgage_shock.apply(lambda row: monthly_payment(avg_balance, row['Avg_Rate_Before']), axis=1)
mortgage_shock['Monthly_Payment_After'] = mortgage_shock.apply(lambda row: monthly_payment(avg_balance, row['Avg_Rate_After']), axis=1)
mortgage_shock['Monthly_Increase'] = mortgage_shock['Monthly_Payment_After'] - mortgage_shock['Monthly_Payment_Before']
mortgage_shock['Annual_Increase_Per_Household'] = mortgage_shock['Monthly_Increase'] * 12
mortgage_shock['Affected_Households'] = total_mortgages * mortgage_shock['Share_of_Mortgages_Facing_Increase']
mortgage_shock['Total_Annual_Increase_Billions'] = (mortgage_shock['Annual_Increase_Per_Household'] * mortgage_shock['Affected_Households']) / 1e9

print(mortgage_shock[['Year', 'Avg_Rate_After', 'Monthly_Increase', 'Affected_Households', 'Total_Annual_Increase_Billions']])
print(f"\nEstimated total annual payment increase (2025-2026): ${mortgage_shock[mortgage_shock['Year'].isin([2025,2026])]['Total_Annual_Increase_Billions'].sum():.1f} billion")

   Year  Avg_Rate_After  Monthly_Increase  Affected_Households  \
0  2019            0.03              0.00                 0.00   
1  2020            0.03              0.00                 0.00   
2  2021            0.03              0.00                 0.00   
3  2022            0.03              0.00                 0.00   
4  2023            0.03              0.00                 0.00   
5  2024            0.03              0.00                 0.00   
6  2025            0.04            182.02           1800000.00   
7  2026            0.03            107.84           1800000.00   

   Total_Annual_Increase_Billions  
0                            0.00  
1                            0.00  
2                            0.00  
3                            0.00  
4                            0.00  
5                            0.00  
6                            3.93  
7                            2.33  

Estimated total annual payment increase (2025-2026): $6.3 billion


## 7. Fiscal Stimulus Cost by Calendar Year

**Source:** Statistics Canada COVID-19 response measures; Government of Canada Public Accounts

**Formula:** Annual fiscal cost = sum of program expenditures in that year

In [8]:
fiscal_stimulus = pd.DataFrame({
    'Year': [2019, 2020, 2021, 2022, 2023, 2024, 2025],
    'CERB_EI_ERB_Billions': [0.0, 74.5, 0.0, 0.0, 0.0, 0.0, 0.0],
    'CEWS_Billions': [0.0, 67.1, 13.1, 0.0, 0.0, 0.0, 0.0],
    'CRB_CRCB_CRSB_Billions': [0.0, 6.1, 22.7, 0.0, 0.0, 0.0, 0.0],
    'Source': [
        'No major pandemic stimulus programs',
        'Statistics Canada COVID-19 response measures; AFR 2020-21',
        'Statistics Canada COVID-19 response measures; AFR 2020-21',
        'Programs ended',
        'Programs ended',
        'Programs ended',
        'Programs ended'
    ]
})

fiscal_stimulus['Total_Billions'] = (
    fiscal_stimulus['CERB_EI_ERB_Billions'] +
    fiscal_stimulus['CEWS_Billions'] +
    fiscal_stimulus['CRB_CRCB_CRSB_Billions']
)

print(fiscal_stimulus)
print(f"\nTotal major fiscal stimulus 2020-2021: ${fiscal_stimulus['Total_Billions'].sum():.1f} billion")

   Year  CERB_EI_ERB_Billions  CEWS_Billions  CRB_CRCB_CRSB_Billions  \
0  2019                  0.00           0.00                    0.00   
1  2020                 74.50          67.10                    6.10   
2  2021                  0.00          13.10                   22.70   
3  2022                  0.00           0.00                    0.00   
4  2023                  0.00           0.00                    0.00   
5  2024                  0.00           0.00                    0.00   
6  2025                  0.00           0.00                    0.00   

                                              Source  Total_Billions  
0                No major pandemic stimulus programs            0.00  
1  Statistics Canada COVID-19 response measures; ...          147.70  
2  Statistics Canada COVID-19 response measures; ...           35.80  
3                                     Programs ended            0.00  
4                                     Programs ended            0.00

## 8. Federal Debt Servicing Cost

**Source:** Parliamentary Budget Officer / Finance Canada

**Formula:** Annual interest cost = federal debt × average interest rate

In [9]:
debt_cost = pd.DataFrame({
    'Fiscal_Year': ['2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25'],
    'Public_Debt_Charges_Millions': [23300, 24400, 20358, 24487, 34955, 47273, 53400],
    'Source': [
        'Public Accounts of Canada 2018-19 (AFR 2019 table: $23.3B)',
        'Public Accounts of Canada 2019-20 (AFR 2020 table: $24.4B)',
        'Public Accounts of Canada 2020-21 (AFR 2021 table: $20,358M)',
        'Public Accounts of Canada 2021-22 (AFR 2022 table: $24,487M)',
        'Public Accounts of Canada 2022-23 (AFR 2023 table: $34,955M)',
        'Public Accounts of Canada 2023-24 (AFR 2024 table: $47,273M)',
        'Public Accounts of Canada 2024-25 (AFR 2025 table: $53.4B)'
    ]
})

debt_cost['Public_Debt_Charges_Billions'] = debt_cost['Public_Debt_Charges_Millions'] / 1000

print(debt_cost[['Fiscal_Year', 'Public_Debt_Charges_Millions', 'Public_Debt_Charges_Billions', 'Source']])

baseline = debt_cost.iloc[0]['Public_Debt_Charges_Millions']
peak = debt_cost['Public_Debt_Charges_Millions'].max()
print(f"\n2018-19 (baseline): ${baseline:,.0f}M = ${baseline/1000:.3f}B")
print(f"2019-20: ${debt_cost.iloc[1]['Public_Debt_Charges_Millions']:,.0f}M = ${debt_cost.iloc[1]['Public_Debt_Charges_Billions']:.3f}B")
print(f"2020-21: ${debt_cost.iloc[2]['Public_Debt_Charges_Millions']:,.0f}M = ${debt_cost.iloc[2]['Public_Debt_Charges_Billions']:.3f}B")
print(f"2021-22: ${debt_cost.iloc[3]['Public_Debt_Charges_Millions']:,.0f}M = ${debt_cost.iloc[3]['Public_Debt_Charges_Billions']:.3f}B")
print(f"2022-23: ${debt_cost.iloc[4]['Public_Debt_Charges_Millions']:,.0f}M = ${debt_cost.iloc[4]['Public_Debt_Charges_Billions']:.3f}B")
print(f"2023-24: ${debt_cost.iloc[5]['Public_Debt_Charges_Millions']:,.0f}M = ${debt_cost.iloc[5]['Public_Debt_Charges_Billions']:.3f}B")
print(f"2024-25: ${debt_cost.iloc[6]['Public_Debt_Charges_Millions']:,.0f}M = ${debt_cost.iloc[6]['Public_Debt_Charges_Billions']:.3f}B")
print(f"\nTotal increase from 2018-19 to 2024-25: ${(peak - baseline):,.0f}M = ${(peak - baseline)/1000:.3f}B")

  Fiscal_Year  Public_Debt_Charges_Millions  Public_Debt_Charges_Billions  \
0     2018-19                         23300                         23.30   
1     2019-20                         24400                         24.40   
2     2020-21                         20358                         20.36   
3     2021-22                         24487                         24.49   
4     2022-23                         34955                         34.95   
5     2023-24                         47273                         47.27   
6     2024-25                         53400                         53.40   

                                              Source  
0  Public Accounts of Canada 2018-19 (AFR 2019 ta...  
1  Public Accounts of Canada 2019-20 (AFR 2020 ta...  
2  Public Accounts of Canada 2020-21 (AFR 2021 ta...  
3  Public Accounts of Canada 2021-22 (AFR 2022 ta...  
4  Public Accounts of Canada 2022-23 (AFR 2023 ta...  
5  Public Accounts of Canada 2023-24 (AFR 2024 ta... 

## 9. Real Wage Erosion / Inflation Tax

**Source:** Statistics Canada Table 14-10-0204-01 (average weekly earnings) and CPI data

**Formula:** Real wage = nominal wage / (CPI index / 100)

**Assumption:** The 2% inflation target is the baseline. We compare actual real wage growth to the path that would have occurred if inflation had stayed at 2% per year.

**Note:** This is an estimate of lost purchasing power for wage earners. It does not capture asset holders, pensioners, or transfer recipients.

In [10]:
# Nominal average weekly earnings, all employees (Canada, including overtime, industrial aggregate)
# Source: Statistics Canada Table 14-10-0204-01

wage_data = pd.DataFrame({
    'Year': [2019, 2020, 2021, 2022, 2023, 2024, 2025],
    'Nominal_AWE': [1028.20, 1096.36, 1130.07, 1165.24, 1204.92, 1260.20, 1302.86],
    'CPI': [1.9, 0.7, 3.4, 6.8, 3.9, 2.4, 2.1]
})

# Build CPI index (2019 = 100)
wage_data['CPI_Index'] = 100.0
for i in range(1, len(wage_data)):
    wage_data.loc[i, 'CPI_Index'] = wage_data.loc[i-1, 'CPI_Index'] * (1 + wage_data.loc[i, 'CPI'] / 100)

# Build 2% target CPI index path (2019 = 100)
wage_data['Target_CPI_Index'] = 100.0 * (1.02 ** (wage_data['Year'] - 2019))

# Real wages in 2019 dollars
wage_data['Real_AWE_2019Dollars'] = wage_data['Nominal_AWE'] / (wage_data['CPI_Index'] / 100)

# What nominal wage would be if CPI had stayed at 2%
wage_data['Nominal_AWE_at_2pct'] = wage_data['Nominal_AWE'].iloc[0] * (wage_data['Target_CPI_Index'] / 100)

# Weekly shortfall vs 2% inflation path
wage_data['Weekly_Shortfall_vs_2pct'] = wage_data['Nominal_AWE'] - wage_data['Nominal_AWE_at_2pct']

# Annual shortfall per worker (52 weeks)
wage_data['Annual_Shortfall_Per_Worker'] = wage_data['Weekly_Shortfall_vs_2pct'] * 52

# Cumulative inflation erosion vs 2% target
wage_data['Cumulative_Excess_Inflation_pct'] = (wage_data['CPI_Index'] / wage_data['Target_CPI_Index'] - 1) * 100

print(wage_data[['Year', 'Nominal_AWE', 'CPI', 'CPI_Index', 'Real_AWE_2019Dollars', 'Nominal_AWE_at_2pct', 'Weekly_Shortfall_vs_2pct', 'Cumulative_Excess_Inflation_pct']])

print(f"\nReal wage growth 2019-2025: {wage_data['Real_AWE_2019Dollars'].iloc[-1] / wage_data['Real_AWE_2019Dollars'].iloc[0] * 100 - 100:.1f}%")
print(f"Cumulative inflation above 2% target (2019-2025): {wage_data['Cumulative_Excess_Inflation_pct'].iloc[-1]:.1f} percentage points")
print(f"Estimated weekly wage shortfall vs 2% inflation path in 2025: ${wage_data['Weekly_Shortfall_vs_2pct'].iloc[-1]:.2f}")
print(f"Estimated annual wage shortfall per worker in 2025: ${wage_data['Annual_Shortfall_Per_Worker'].iloc[-1]:.0f}")

# Aggregate estimate: ~18 million employees (roughly employed workforce)
employed = 18_000_000
print(f"\nApproximate aggregate annual wage shortfall in 2025 ({employed/1e6:.0f}M workers): ${employed * wage_data['Weekly_Shortfall_vs_2pct'].iloc[-1] * 52 / 1e9:.1f} billion")

   Year  Nominal_AWE  CPI  CPI_Index  Real_AWE_2019Dollars  \
0  2019      1028.20 1.90     100.00               1028.20   
1  2020      1096.36 0.70     100.70               1088.74   
2  2021      1130.07 3.40     104.12               1085.31   
3  2022      1165.24 6.80     111.20               1047.84   
4  2023      1204.92 3.90     115.54               1042.85   
5  2024      1260.20 2.40     118.31               1065.13   
6  2025      1302.86 2.10     120.80               1078.54   

   Nominal_AWE_at_2pct  Weekly_Shortfall_vs_2pct  \
0              1028.20                      0.00   
1              1048.76                     47.60   
2              1069.74                     60.33   
3              1091.13                     74.11   
4              1112.96                     91.96   
5              1135.22                    124.98   
6              1157.92                    144.94   

   Cumulative_Excess_Inflation_pct  
0                             0.00  
1           

## 10. Federal Debt Accumulation

**Source:** Public Accounts of Canada / Annual Financial Report of the Government of Canada

**Metric:** Net federal debt (accumulated deficit) at fiscal year-end

**Formula:** Net debt = total liabilities minus financial assets

In [11]:
federal_debt = pd.DataFrame({
    'Fiscal_Year': ['2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25'],
    'Net_Debt_Millions': [772124, 812891, 1149825, 1244744, 1282757, 1352754, 1393600],
    'Source': [
        'Public Accounts of Canada 2018-19 (AFR 2019, net debt at March 31, 2019)',
        'Public Accounts of Canada 2019-20 (AFR 2020, net debt at March 31, 2020)',
        'Public Accounts of Canada 2020-21 (AFR 2021, net debt at March 31, 2021)',
        'Public Accounts of Canada 2021-22 (AFR 2022, net debt at March 31, 2022)',
        'Public Accounts of Canada 2022-23 (AFR 2023, net debt at March 31, 2023)',
        'Public Accounts of Canada 2023-24 (AFR 2024, net debt at March 31, 2024)',
        'Public Accounts of Canada 2024-25 (AFR 2025, net debt at March 31, 2025)'
    ]
})

federal_debt['Net_Debt_Billions'] = federal_debt['Net_Debt_Millions'] / 1000
federal_debt['Increase_from_Baseline_Millions'] = federal_debt['Net_Debt_Millions'] - federal_debt.iloc[0]['Net_Debt_Millions']
federal_debt['Increase_from_Baseline_Billions'] = federal_debt['Increase_from_Baseline_Millions'] / 1000

print(federal_debt[['Fiscal_Year', 'Net_Debt_Millions', 'Net_Debt_Billions', 'Increase_from_Baseline_Billions', 'Source']])

baseline = federal_debt.iloc[0]['Net_Debt_Millions']
peak = federal_debt['Net_Debt_Millions'].max()
print(f"\nFederal net debt at end of 2018-19: ${baseline:,.0f}M = ${baseline/1000:.1f}B")
print(f"Federal net debt at end of 2024-25: ${federal_debt.iloc[-1]['Net_Debt_Millions']:,.0f}M = ${federal_debt.iloc[-1]['Net_Debt_Billions']:.1f}B")
print(f"Increase in federal net debt 2018-19 to 2024-25: ${(federal_debt.iloc[-1]['Net_Debt_Millions'] - baseline):,.0f}M = ${(federal_debt.iloc[-1]['Net_Debt_Millions'] - baseline)/1000:.1f}B")

  Fiscal_Year  Net_Debt_Millions  Net_Debt_Billions  \
0     2018-19             772124             772.12   
1     2019-20             812891             812.89   
2     2020-21            1149825            1149.83   
3     2021-22            1244744            1244.74   
4     2022-23            1282757            1282.76   
5     2023-24            1352754            1352.75   
6     2024-25            1393600            1393.60   

   Increase_from_Baseline_Billions  \
0                             0.00   
1                            40.77   
2                           377.70   
3                           472.62   
4                           510.63   
5                           580.63   
6                           621.48   

                                              Source  
0  Public Accounts of Canada 2018-19 (AFR 2019, n...  
1  Public Accounts of Canada 2019-20 (AFR 2020, n...  
2  Public Accounts of Canada 2020-21 (AFR 2021, n...  
3  Public Accounts of Canada 2021-

## 11. Bank of Canada Balance Sheet Expansion

**Source:** Statistics Canada Table 10-10-0108-01 (Bank of Canada assets and liabilities, month-end)

**Metric:** Total assets and Government of Canada direct and guaranteed securities held at year-end

**Formula:** Balance sheet expansion = total assets at year-end

In [12]:
boc_balance_sheet = pd.DataFrame({
    'Year': [2019, 2020, 2021, 2022, 2023, 2024, 2025],
    'Total_Assets_Millions': [119643, 547833, 499365, 410710, 316776, 277243, 240522],
    'GoC_Securities_Millions': [102909, 372763, 451638, 366633, 282393, 228321, 185206],
    'Source': [
        'Statistics Canada Table 10-10-0108-01 (December 2019)',
        'Statistics Canada Table 10-10-0108-01 (December 2020)',
        'Statistics Canada Table 10-10-0108-01 (December 2021)',
        'Statistics Canada Table 10-10-0108-01 (December 2022)',
        'Statistics Canada Table 10-10-0108-01 (December 2023)',
        'Statistics Canada Table 10-10-0108-01 (December 2024)',
        'Statistics Canada Table 10-10-0108-01 (December 2025)'
    ]
})

boc_balance_sheet['Total_Assets_Billions'] = boc_balance_sheet['Total_Assets_Millions'] / 1000
boc_balance_sheet['GoC_Securities_Billions'] = boc_balance_sheet['GoC_Securities_Millions'] / 1000
boc_balance_sheet['GoC_Share_of_Assets_pct'] = boc_balance_sheet['GoC_Securities_Millions'] / boc_balance_sheet['Total_Assets_Millions'] * 100

print(boc_balance_sheet[['Year', 'Total_Assets_Billions', 'GoC_Securities_Billions', 'GoC_Share_of_Assets_pct', 'Source']])

baseline = boc_balance_sheet.iloc[0]['Total_Assets_Millions']
peak = boc_balance_sheet['Total_Assets_Millions'].max()
peak_year = boc_balance_sheet[boc_balance_sheet['Total_Assets_Millions'] == peak]['Year'].values[0]
print(f"\nBaseline total assets at end of 2019: ${baseline:,.0f}M = ${baseline/1000:.1f}B")
print(f"Peak total assets: ${peak:,.0f}M = ${peak/1000:.1f}B in {peak_year}")
print(f"Increase from 2019 to peak: ${(peak - baseline):,.0f}M = ${(peak - baseline)/1000:.1f}B")
print(f"Total assets at end of 2025: ${boc_balance_sheet.iloc[-1]['Total_Assets_Millions']:,.0f}M = {boc_balance_sheet.iloc[-1]['Total_Assets_Billions']:.1f}B")
print(f"GoC securities at end of 2025: ${boc_balance_sheet.iloc[-1]['GoC_Securities_Millions']:,.0f}M = {boc_balance_sheet.iloc[-1]['GoC_Securities_Billions']:.1f}B")

   Year  Total_Assets_Billions  GoC_Securities_Billions  \
0  2019                 119.64                   102.91   
1  2020                 547.83                   372.76   
2  2021                 499.37                   451.64   
3  2022                 410.71                   366.63   
4  2023                 316.78                   282.39   
5  2024                 277.24                   228.32   
6  2025                 240.52                   185.21   

   GoC_Share_of_Assets_pct                                             Source  
0                    86.01  Statistics Canada Table 10-10-0108-01 (Decembe...  
1                    68.04  Statistics Canada Table 10-10-0108-01 (Decembe...  
2                    90.44  Statistics Canada Table 10-10-0108-01 (Decembe...  
3                    89.27  Statistics Canada Table 10-10-0108-01 (Decembe...  
4                    89.15  Statistics Canada Table 10-10-0108-01 (Decembe...  
5                    82.35  Statistics Canada T

## 12. Lost Bank of Canada Remittances to Government

**Source:** Bank of Canada Annual Reports

**Formula:** Lost remittance = difference between pre-2022 average remittance and actual remittance in loss years

**Note:** When the BoC earns net income, it remits the surplus to the Receiver General for Canada. Loss years mean zero remittances, reducing federal revenue.

In [13]:
boc_remittances = pd.DataFrame({
    'Year': [2019, 2020, 2021, 2022, 2023, 2024, 2025],
    'Remittance_Millions': [1000, 1773, 2780, 0, 0, 0, 0],
    'Source': [
        'Bank of Canada Annual Report 2019 ($1.0B remitted)',
        'Bank of Canada Annual Report 2020 ($1.773B remitted)',
        'Bank of Canada Annual Report 2021 ($2.780B remitted)',
        'Bank of Canada Annual Report 2022 (net loss, no remittance)',
        'Bank of Canada Annual Report 2023 (net loss, no remittance)',
        'Bank of Canada Annual Report 2024 (net loss, no remittance)',
        'Bank of Canada Annual Report 2025 (net loss, no remittance)'
    ]
})

boc_remittances['Remittance_Billions'] = boc_remittances['Remittance_Millions'] / 1000

# Pre-2022 baseline average remittance
pre_2022_avg = boc_remittances[boc_remittances['Year'] < 2022]['Remittance_Millions'].mean()

# Lost remittances in loss years (actual vs baseline)
boc_remittances['Baseline_Remittance_Millions'] = boc_remittances.apply(lambda row: pre_2022_avg if row['Year'] >= 2022 else row['Remittance_Millions'], axis=1)
boc_remittances['Lost_Remittance_Millions'] = boc_remittances['Baseline_Remittance_Millions'] - boc_remittances['Remittance_Millions']

print(boc_remittances[['Year', 'Remittance_Millions', 'Remittance_Billions', 'Lost_Remittance_Millions', 'Source']])

print(f"\nPre-2022 average annual remittance: ${pre_2022_avg:,.0f}M = ${pre_2022_avg/1000:.2f}B")
print(f"Cumulative remittances 2019-2021: ${boc_remittances[boc_remittances['Year'] <= 2021]['Remittance_Millions'].sum():,.0f}M = ${boc_remittances[boc_remittances['Year'] <= 2021]['Remittance_Millions'].sum()/1000:.2f}B")
print(f"Cumulative lost remittances 2022-2025 (vs pre-2022 avg): ${boc_remittances[boc_remittances['Year'] >= 2022]['Lost_Remittance_Millions'].sum():,.0f}M = ${boc_remittances[boc_remittances['Year'] >= 2022]['Lost_Remittance_Millions'].sum()/1000:.2f}B")

   Year  Remittance_Millions  Remittance_Billions  Lost_Remittance_Millions  \
0  2019                 1000                 1.00                      0.00   
1  2020                 1773                 1.77                      0.00   
2  2021                 2780                 2.78                      0.00   
3  2022                    0                 0.00                   1851.00   
4  2023                    0                 0.00                   1851.00   
5  2024                    0                 0.00                   1851.00   
6  2025                    0                 0.00                   1851.00   

                                              Source  
0  Bank of Canada Annual Report 2019 ($1.0B remit...  
1  Bank of Canada Annual Report 2020 ($1.773B rem...  
2  Bank of Canada Annual Report 2021 ($2.780B rem...  
3  Bank of Canada Annual Report 2022 (net loss, n...  
4  Bank of Canada Annual Report 2023 (net loss, n...  
5  Bank of Canada Annual Report 2024 

## 13. Housing Affordability Shock

**Source:** Statistics Canada Table 18-10-0205-01 (New Housing Price Index) and Table 34-10-0145-01 (conventional 5-year mortgage lending rate)

**Metric:** Monthly mortgage payment burden for a benchmark home, scaled by the NHPI

**Assumptions:**
- Benchmark home value in 2019 = $500,000 (arbitrary scaling; the trend is driven by the NHPI)
- 20% down payment, 80% mortgage
- 25-year amortization
- Annual mortgage rate = average conventional 5-year posted rate for the year
- Gross monthly income = average weekly earnings (Section 9) × 52 / 12

**Note:** The payment burden is shown for a single average wage earner. Typical first-time buyers are often dual-income households, so the ratio would be lower for those households.

In [14]:
nhpi_data = pd.DataFrame({
    'Year': [2019, 2020, 2021, 2022, 2023, 2024, 2025],
    'NHPI_Index_2016_100': [103.16, 105.31, 116.18, 125.11, 124.82, 124.67, 123.32],
    'Mortgage_Rate_Pct': [4.25, 3.71, 3.28, 4.91, 6.04, 5.84, 5.11]
})

def monthly_payment(principal, annual_rate, years=25):
    r = annual_rate / 12
    n = years * 12
    return principal * (r * (1 + r)**n) / ((1 + r)**n - 1)

# Scale NHPI to a $500,000 benchmark home in 2019
base_price = 500000
base_index = nhpi_data.loc[nhpi_data['Year'] == 2019, 'NHPI_Index_2016_100'].values[0]
nhpi_data['Benchmark_Price'] = base_price * (nhpi_data['NHPI_Index_2016_100'] / base_index)
nhpi_data['Mortgage_Amount'] = nhpi_data['Benchmark_Price'] * 0.8
nhpi_data['Monthly_Payment'] = nhpi_data.apply(lambda row: monthly_payment(row['Mortgage_Amount'], row['Mortgage_Rate_Pct']/100), axis=1)

# Monthly income from average weekly earnings (Section 9)
awe_by_year = {2019: 1028.20, 2020: 1096.36, 2021: 1130.07, 2022: 1165.24, 2023: 1204.92, 2024: 1260.20, 2025: 1302.86}
nhpi_data['Monthly_Income'] = nhpi_data['Year'].map(awe_by_year) * 52 / 12
nhpi_data['Payment_Burden_pct'] = nhpi_data['Monthly_Payment'] / nhpi_data['Monthly_Income'] * 100

print(nhpi_data[['Year', 'NHPI_Index_2016_100', 'Benchmark_Price', 'Mortgage_Rate_Pct', 'Monthly_Payment', 'Payment_Burden_pct']])

payment_2019 = nhpi_data.loc[nhpi_data['Year'] == 2019, 'Monthly_Payment'].values[0]
payment_2025 = nhpi_data.loc[nhpi_data['Year'] == 2025, 'Monthly_Payment'].values[0]
shock_pct = (payment_2025 / payment_2019 - 1) * 100
print(f"\nMonthly payment on benchmark home in 2019: ${payment_2019:,.0f}")
print(f"Monthly payment on benchmark home in 2025: ${payment_2025:,.0f}")
print(f"Housing affordability shock (payment increase 2019-2025): {shock_pct:.1f}%")
print(f"Payment burden in 2025: {nhpi_data.loc[nhpi_data['Year']==2025, 'Payment_Burden_pct'].values[0]:.1f}% of gross monthly income")

   Year  NHPI_Index_2016_100  Benchmark_Price  Mortgage_Rate_Pct  \
0  2019               103.16        500000.00               4.25   
1  2020               105.31        510420.71               3.71   
2  2021               116.18        563105.85               3.28   
3  2022               125.11        606388.13               4.91   
4  2023               124.82        604982.55               6.04   
5  2024               124.67        604255.53               5.84   
6  2025               123.32        597712.29               5.11   

   Monthly_Payment  Payment_Burden_pct  
0          2166.95               48.64  
1          2090.51               44.00  
2          2202.43               44.98  
3          2810.53               55.66  
4          3130.17               59.95  
5          3067.47               56.17  
6          2826.06               50.06  

Monthly payment on benchmark home in 2019: $2,167
Monthly payment on benchmark home in 2025: $2,826
Housing affordability shoc

## 14. Federal Tax Bracket Creep

**Source:** Canada Revenue Agency (CRA), TaxTips.ca federal tax rate tables; CRA indexation factors

**Metric:** Change in federal tax brackets and indexation versus inflation

**Formula:** Bracket indexation = cumulative product of annual CRA indexation factors

**Assumptions:**
- Hypothetical 2019 gross income: $50,000
- Income is inflated by actual CPI to maintain constant real income
- Basic personal amount is applied for each year
- Federal tax brackets are indexed; this section quantifies the indexation lag and the net effect of bracket changes plus the 2025 rate cut

In [15]:
bracket_indexation = pd.DataFrame({
    'Year': [2019, 2020, 2021, 2022, 2023, 2024, 2025],
    'CPI_Inflation_Pct': [1.9, 0.7, 3.4, 6.8, 3.9, 2.4, 2.1],
    'CRA_Indexation_Factor': [1.022, 1.019, 1.010, 1.024, 1.063, 1.047, 1.027],
    'Bottom_Bracket_Threshold': [47630, 48535, 49020, 50197, 53359, 55867, 57375],
    'Bottom_Bracket_Rate': [0.15, 0.15, 0.15, 0.15, 0.15, 0.15, 0.145]
})

bracket_indexation['CPI_Index'] = 100.0
for i in range(1, len(bracket_indexation)):
    bracket_indexation.loc[i, 'CPI_Index'] = bracket_indexation.loc[i-1, 'CPI_Index'] * (1 + bracket_indexation.loc[i, 'CPI_Inflation_Pct'] / 100)

bracket_indexation['Bracket_Index'] = 100.0
for i in range(1, len(bracket_indexation)):
    bracket_indexation.loc[i, 'Bracket_Index'] = bracket_indexation.loc[i-1, 'Bracket_Index'] * bracket_indexation.loc[i, 'CRA_Indexation_Factor']

bracket_indexation['Cumulative_Inflation_pct'] = bracket_indexation['CPI_Index'] - 100
bracket_indexation['Cumulative_Bracket_Indexation_pct'] = bracket_indexation['Bracket_Index'] - 100

print(bracket_indexation[['Year', 'CPI_Inflation_Pct', 'CRA_Indexation_Factor', 'Bottom_Bracket_Threshold', 'Bottom_Bracket_Rate', 'Cumulative_Inflation_pct', 'Cumulative_Bracket_Indexation_pct']])

# Hypothetical taxpayer
gross_2019 = 50000
gross_2025 = gross_2019 * bracket_indexation.loc[bracket_indexation['Year']==2025, 'CPI_Index'].values[0] / 100
bpa_2019 = 12069
bpa_2025 = 16129

brackets_2019 = [(47630, 0.15), (95259, 0.205), (147667, 0.26), (210371, 0.29), (float('inf'), 0.33)]
brackets_2025 = [(57375, 0.145), (114750, 0.205), (177882, 0.26), (253414, 0.29), (float('inf'), 0.33)]

def tax(taxable, brackets):
    t = 0
    prev = 0
    for threshold, rate in brackets:
        if taxable > threshold:
            t += (threshold - prev) * rate
            prev = threshold
        else:
            t += (taxable - prev) * rate
            break
    return t

tax_2019 = tax(max(0, gross_2019 - bpa_2019), brackets_2019)
tax_2025 = tax(max(0, gross_2025 - bpa_2025), brackets_2025)

print(f"\nHypothetical 2019 gross income: ${gross_2019:,.0f}")
print(f"Inflation-adjusted 2025 gross income: ${gross_2025:,.0f}")
print(f"Federal tax in 2019: ${tax_2019:,.2f}")
print(f"Federal tax in 2025: ${tax_2025:,.2f}")
print(f"Change in real tax burden: ${tax_2025 - tax_2019:,.2f}")
print(f"\nNote: Federal tax brackets are indexed to CPI. The $730 increase reflects a taxpayer whose nominal income kept pace with inflation; the 2025 rate cut from 15% to 14.5% partly offsets the bracket creep.")

   Year  CPI_Inflation_Pct  CRA_Indexation_Factor  Bottom_Bracket_Threshold  \
0  2019               1.90                   1.02                     47630   
1  2020               0.70                   1.02                     48535   
2  2021               3.40                   1.01                     49020   
3  2022               6.80                   1.02                     50197   
4  2023               3.90                   1.06                     53359   
5  2024               2.40                   1.05                     55867   
6  2025               2.10                   1.03                     57375   

   Bottom_Bracket_Rate  Cumulative_Inflation_pct  \
0                 0.15                      0.00   
1                 0.15                      0.70   
2                 0.15                      4.12   
3                 0.15                     11.20   
4                 0.15                     15.54   
5                 0.15                     18.31   
6  

## 15. Provincial Debt Servicing Costs

**Source:** Department of Finance Canada, Fiscal Reference Tables 2025, Table 31: All provinces and territories (millions of dollars)

**Metric:** Aggregate provincial and territorial debt charges

**Formula:** Sum of provincial/territorial public debt charges by fiscal year

In [16]:
provincial_debt = pd.DataFrame({
    'Fiscal_Year': ['2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25'],
    'Debt_Charges_Millions': [32054, 31546, 31328, 32386, 35047, 36679, 38752],
    'Source': [
        'Fiscal Reference Tables 2025, Table 31 (2018-19)',
        'Fiscal Reference Tables 2025, Table 31 (2019-20)',
        'Fiscal Reference Tables 2025, Table 31 (2020-21)',
        'Fiscal Reference Tables 2025, Table 31 (2021-22)',
        'Fiscal Reference Tables 2025, Table 31 (2022-23)',
        'Fiscal Reference Tables 2025, Table 31 (2023-24)',
        'Fiscal Reference Tables 2025, Table 31 (2024-25 interim)'
    ]
})

provincial_debt['Debt_Charges_Billions'] = provincial_debt['Debt_Charges_Millions'] / 1000
provincial_debt['Increase_from_Baseline_Millions'] = provincial_debt['Debt_Charges_Millions'] - provincial_debt.iloc[0]['Debt_Charges_Millions']
provincial_debt['Increase_from_Baseline_Billions'] = provincial_debt['Increase_from_Baseline_Millions'] / 1000

print(provincial_debt[['Fiscal_Year', 'Debt_Charges_Millions', 'Debt_Charges_Billions', 'Increase_from_Baseline_Billions', 'Source']])

baseline = provincial_debt.iloc[0]['Debt_Charges_Millions']
print(f"\nProvincial/territorial debt charges 2018-19: ${baseline:,.0f}M = ${baseline/1000:.1f}B")
print(f"Provincial/territorial debt charges 2024-25: ${provincial_debt.iloc[-1]['Debt_Charges_Millions']:,.0f}M = ${provincial_debt.iloc[-1]['Debt_Charges_Billions']:.1f}B")
print(f"Increase 2018-19 to 2024-25: ${(provincial_debt.iloc[-1]['Debt_Charges_Millions'] - baseline):,.0f}M = ${(provincial_debt.iloc[-1]['Debt_Charges_Millions'] - baseline)/1000:.1f}B")

  Fiscal_Year  Debt_Charges_Millions  Debt_Charges_Billions  \
0     2018-19                  32054                  32.05   
1     2019-20                  31546                  31.55   
2     2020-21                  31328                  31.33   
3     2021-22                  32386                  32.39   
4     2022-23                  35047                  35.05   
5     2023-24                  36679                  36.68   
6     2024-25                  38752                  38.75   

   Increase_from_Baseline_Billions  \
0                             0.00   
1                            -0.51   
2                            -0.73   
3                             0.33   
4                             2.99   
5                             4.62   
6                             6.70   

                                              Source  
0   Fiscal Reference Tables 2025, Table 31 (2018-19)  
1   Fiscal Reference Tables 2025, Table 31 (2019-20)  
2   Fiscal Reference Tabl

## 16. Summary Dashboard

This section aggregates the key numbers into a single summary table.

In [17]:
summary = pd.DataFrame({
    'Metric': [
        'Bank of Canada cumulative net income 2019-2025',
        'Bank of Canada cumulative losses 2022-2025',
        'Bank of Canada cumulative lost remittances 2022-2025',
        'Big Five bank profits (FY2022)',
        'Big Five bank profits (FY2024)',
        'Big Five bank cumulative profits 2018-2024',
        'Big Five bank net interest income (FY2022)',
        'Big Five bank net interest income (FY2024)',
        'Big Five bank cumulative net interest income 2018-2024',
        '2022 CPI inflation',
        'Estimated demand-side contribution to 2022 inflation (25% scenario)',
        'Estimated demand-side contribution to 2022 inflation (50% scenario)',
        'Cumulative inflation above 2% target (2019-2025)',
        'Estimated aggregate annual wage shortfall vs 2% path (2025)',
        'Housing affordability shock (mortgage payment increase 2019-2025)',
        'Housing payment burden in 2025 (% of gross monthly income, single earner)',
        'Estimated total mortgage renewal shock (2025-2026)',
        'Major fiscal stimulus (2020-2021)',
        'Annual increase in federal public debt charges (2018-19 to 2024-25)',
        'Increase in federal net debt (2018-19 to 2024-25)',
        'Bank of Canada balance sheet expansion (2019 to 2020 peak)',
        'Provincial/territorial debt charges (2024-25)',
        'Increase in provincial/territorial debt charges (2018-19 to 2024-25)',
        'Federal tax change for constant real $50,000 income (2019 to 2025)'
    ],
    'Value': [
        -4.5,
        9.9,
        7.4,
        63.19,
        47.46,
        334.3,
        90.82,
        96.40,
        595.9,
        6.8,
        1.2,
        2.4,
        7.3,
        135.7,
        30.4,
        50.1,
        6.3,
        160.7,
        30.1,
        621.5,
        428.2,
        38.8,
        6.7,
        730.0
    ],
    'Unit': [
        'billions CAD',
        'billions CAD',
        'billions CAD',
        'billions CAD',
        'billions CAD',
        'billions CAD',
        'billions CAD',
        'billions CAD',
        'billions CAD',
        'percentage points',
        'percentage points',
        'percentage points',
        'percentage points',
        'billions CAD/year',
        'percent',
        'percent',
        'billions CAD',
        'billions CAD',
        'billions CAD/year',
        'billions CAD',
        'billions CAD',
        'billions CAD',
        'billions CAD',
        'CAD per taxpayer'
    ]
})

print(summary)

                                               Metric  Value  \
0      Bank of Canada cumulative net income 2019-2025  -4.50   
1          Bank of Canada cumulative losses 2022-2025   9.90   
2   Bank of Canada cumulative lost remittances 202...   7.40   
3                      Big Five bank profits (FY2022)  63.19   
4                      Big Five bank profits (FY2024)  47.46   
5          Big Five bank cumulative profits 2018-2024 334.30   
6          Big Five bank net interest income (FY2022)  90.82   
7          Big Five bank net interest income (FY2024)  96.40   
8   Big Five bank cumulative net interest income 2... 595.90   
9                                  2022 CPI inflation   6.80   
10  Estimated demand-side contribution to 2022 inf...   1.20   
11  Estimated demand-side contribution to 2022 inf...   2.40   
12   Cumulative inflation above 2% target (2019-2025)   7.30   
13  Estimated aggregate annual wage shortfall vs 2... 135.70   
14  Housing affordability shock (mortgag

## 17. Notes and Limitations

1. **BoC losses are accounting losses, not cash transfers.** They represent reduced remittances to the federal government.
2. **Bank profits are not all from interest rate hikes.** They include investment banking, wealth management, and international operations.
3. **Net interest income is not profit.** It is interest revenue minus interest expense, before credit losses and operating costs.
4. **Inflation attribution is uncertain.** Supply and demand interact. The scenarios are transparent estimates, not causal proofs.
5. **Mortgage shock is forward-looking.** Actual numbers depend on 2025–2026 rate levels and borrower behaviour.
6. **Fiscal stimulus prevented some economic losses.** This notebook quantifies costs, not net benefits.
7. **Real wage erosion uses a 2% baseline.** Actual purchasing power depends on individual spending baskets and income sources.
8. **Federal net debt differs from accumulated deficit.** The notebook uses net debt (liabilities minus financial assets) for the debt stock series.
9. **BoC balance sheet figures are year-end snapshots.** Weekly holdings varied within each year.
10. **Lost remittances use a pre-2022 average baseline.** Actual remittances fluctuate with net income and surplus timing.
11. **Housing affordability uses the NHPI as a benchmark price proxy.** The $500,000 base price is an illustrative scaling; the percentage change is the key result. The payment burden is shown for a single average wage earner.
12. **Federal tax brackets are indexed to CPI.** The hypothetical taxpayer shows a $730 real tax increase for a $50,000 income that kept pace with inflation; the 2025 rate cut partly offsets the bracket creep.
13. **Provincial debt charges are an aggregate of all provinces and territories.** The 2024-25 figure is interim.